# Duka Sales Exploration

ADEN AHMED MOHAMED — Cohort 1M DEVIS

Duka Electronics half-year sales review, January–June 2026.

## Executive Summary

- The 1,000 orders generated KES 71,792,500 in revenue during the supplied January–June 2026 period.
- Nairobi led with KES 26,106,100, or 36.36% of revenue, and 376 orders, so it should remain the main sales focus.
- Laptops generated KES 41,535,000, while Tablets led unit sales at 691, supporting different priorities for revenue and volume.
- April was the best month at KES 13,518,300, exceeding June's KES 10,903,400 by 23.98%, using the worst month as the baseline.
- The Premium Online segment in Nairobi and Mombasa contained 83 orders worth KES 10,218,000, or 14.23% of total revenue, making it a specific segment for follow-up.

## Task 1: First-five-minutes inspection

Load dates at ingestion, then inspect shape, first/last records, types, missingness, descriptive statistics, uniqueness, category shares and the actual date range before calculating business metrics.

In [1]:
from pathlib import Path
import runpy
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

# Run from the repository root. Regenerate the supplied raw data if absent.
if not Path("duka_sales.csv").exists():
    runpy.run_path("generate_data.py")
sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
pd.set_option("display.max_columns", 15)
print("Shape:", sales.shape)
print("First five records:")
display(sales.head())
print("Last five records:")
display(sales.tail())
sales.info()
display(sales.describe(include="all"))
display(sales.isna().sum().rename("missing_values").to_frame())
display(sales.nunique().rename("distinct_values").to_frame())
print("Date range:", sales.order_date.min(), "to", sales.order_date.max())
print("Branch proportions:")
display(sales.branch.value_counts(normalize=True).rename("order_share").to_frame())
print("Channel proportions:")
display(sales.channel.value_counts(normalize=True).rename("order_share").to_frame())
assert sales.order_id.is_unique
assert len(sales) == 1000
assert not sales.isna().any().any()

Shape: (1000, 7)
First five records:


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


Last five records:


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


,order_id,order_date,branch,product,quantity,channel,unit_price
count,1000.000000,1000,1000,1000,1000.000000,1000,1000.000000
unique,NaN,NaN,5,5,NaN,3,NaN
top,NaN,NaN,Nairobi,Laptop,NaN,In-store,NaN
freq,NaN,NaN,376,220,NaN,579,NaN
mean,1500.500000,2026-03-30 08:44:09.600000,NaN,NaN,3.012000,NaN,23918.300000
min,1001.000000,2026-01-01 00:00:00,NaN,NaN,1.000000,NaN,1200.000000
25%,1250.750000,2026-02-14 00:00:00,NaN,NaN,2.000000,NaN,2500.000000
50%,1500.500000,2026-03-29 00:00:00,NaN,NaN,3.000000,NaN,15000.000000
75%,1750.250000,2026-05-14 00:00:00,NaN,NaN,4.000000,NaN,28000.000000
max,2000.000000,2026-06-29 00:00:00,NaN,NaN,5.000000,NaN,65000.000000


... (2 outputs left out)

Date range: 2026-01-01 00:00:00 to 2026-06-29 00:00:00
Branch proportions:


,order_share
branch,
Nairobi,0.376
Mombasa,0.186
Nakuru,0.162
Kisumu,0.159
Eldoret,0.117


Channel proportions:


,order_share
channel,
In-store,0.579
Online,0.320
Phone order,0.101


### Inspection observations

- **head():** the first five records expose one order per row, with product, quantity, channel and unit price available for revenue calculation.
- **tail():** the final five records retain the same seven-column schema and the order-ID sequence ends at 2000.
- **shape:** there are 1,000 rows and 7 original columns (before derived revenue in this explanatory calculation).
- **info():** order_date is datetime, order_id/quantity/unit_price are integers, and branch/product/channel are categorical text; every original column has 1,000 non-null values.
- **describe():** quantity ranges from 1 to 5 and unit prices range from KES 1,200 to KES 65,000, so order count alone need not determine revenue.
- **missing values and uniqueness:** no source values are missing and all 1,000 order IDs are unique.
- **categories:** five branches, five products and three channels occur; Nairobi supplies 37.6% of orders and In-store supplies 57.9%.
- **date range:** 01 January 2026 to 29 June 2026; the generator samples 180 days starting 1 January, so June is not a complete calendar month.

The uploaded Practicals include illustrative outputs; all findings here are recalculated from the unchanged seed-2026 generator.

## Task 2: Derived columns

Revenue is quantity × unit_price; month and weekday use datetime accessors; weekend means Saturday/Sunday; Premium refers to unit price of at least KES 28,000, rather than the order total.

In [2]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek.ge(5)
sales["price_band"] = np.where(sales["unit_price"].ge(28000), "Premium", "Value")
display(sales.head())
for column in ["month", "weekday", "is_weekend", "price_band"]:
    display(sales[column].value_counts().rename("orders").to_frame())

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value


,orders
month,
March,186
January,173
April,167
June,162
February,159
May,153


,orders
weekday,
Friday,166
Thursday,145
Sunday,145
Monday,140
Wednesday,137
Tuesday,135
Saturday,132


,orders
is_weekend,
False,723
True,277


,orders
price_band,
Value,565
Premium,435


## Task 3: Rank branches by revenue and orders

Aggregate once by branch, then compare the descending revenue and order rankings; average order revenue helps explain the relationship.

In [3]:
branch_summary = sales.groupby("branch").agg(
    total_revenue=("revenue", "sum"), orders=("order_id", "size"),
    average_order_revenue=("revenue", "mean"), units=("quantity", "sum")
)
revenue_ranking = branch_summary.sort_values("total_revenue", ascending=False)
order_ranking = branch_summary.sort_values("orders", ascending=False)
display(revenue_ranking.round(2))
display(order_ranking[["orders", "total_revenue"]])
print("Identical branch ordering:", revenue_ranking.index.equals(order_ranking.index))

,total_revenue,orders,average_order_revenue,units
branch,,,,
Nairobi,26106100,376,69431.12,1107
Mombasa,14003700,186,75288.71,567
Nakuru,11970400,162,73891.36,495
Kisumu,10868100,159,68352.83,473
Eldoret,8844200,117,75591.45,370


,orders,total_revenue
branch,,
Nairobi,376,26106100
Mombasa,186,14003700
Nakuru,162,11970400
Kisumu,159,10868100
Eldoret,117,8844200


Identical branch ordering: True


**Answer:** Both rankings are Nairobi, Mombasa, Nakuru, Kisumu and Eldoret. Nairobi has KES 26,106,100 revenue and 376 orders, compared with Eldoret’s KES 8,844,200 and 117 orders. Average order revenue varies from KES 68,352.83 to KES 75,591.45, but those differences do not offset the order-volume gaps in this sample; revenue is not guaranteed to follow order counts in other periods.

## Task 4: Top products by revenue and units

Sum revenue and units separately, then take the largest three in each measure.

In [4]:
product_summary = sales.groupby("product").agg(
    revenue=("revenue", "sum"), units=("quantity", "sum")
)
top_revenue = product_summary["revenue"].nlargest(3)
top_units = product_summary["units"].nlargest(3)
display(top_revenue.to_frame())
display(top_units.to_frame())

,revenue
product,
Laptop,41535000
Tablet,19348000
Phone,8880000


,units
product,
Tablet,691
Laptop,639
Phone,592


**Answer:** Revenue ranks Laptop (KES 41,535,000), Tablet (KES 19,348,000) and Phone (KES 8,880,000). Units rank Tablet (691), Laptop (639) and Phone (592). The same three products appear, but Tablet and Laptop exchange positions because a Laptop costs KES 65,000 against a Tablet’s KES 28,000; revenue combines price and volume.

## Task 5: Weekday and weekend comparison

Group by is_weekend for orders, total revenue and average order revenue; no difference in totals should be interpreted as daily productivity without accounting for days.

In [5]:
weekend_comparison = sales.groupby("is_weekend").agg(
    orders=("order_id", "size"), total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean")
).rename(index={False: "Weekday", True: "Weekend"})
display(weekend_comparison.round(2))
weekend_aov_difference = (weekend_comparison.loc["Weekend", "average_order_revenue"]
                          / weekend_comparison.loc["Weekday", "average_order_revenue"] - 1) * 100
print(f"Weekend average order revenue above weekday: {weekend_aov_difference:.2f}%")

,orders,total_revenue,average_order_revenue
is_weekend,,,
Weekday,723,51856200,71723.65
Weekend,277,19936300,71972.20


Weekend average order revenue above weekday: 0.35%


**Answer:** Weekdays have 723 orders and KES 51,856,200 revenue, averaging KES 71,723.65 per order. Weekends have 277 orders and KES 19,936,300 revenue, averaging KES 71,972.20; their average order revenue is only 0.35% higher. The much larger weekday revenue comes from order volume, not a meaningful basket-value gap, and weekdays occur more often than weekend days.

## Task 6: Best and worst revenue months

Group by calendar month using a PeriodIndex and compare maximum with minimum; percentage difference is (best − worst) ÷ worst × 100.

In [6]:
monthly_revenue = sales.groupby(sales["order_date"].dt.to_period("M"))["revenue"].sum()
best_month, worst_month = monthly_revenue.idxmax(), monthly_revenue.idxmin()
month_difference_pct = (monthly_revenue.max() - monthly_revenue.min()) / monthly_revenue.min() * 100
display(monthly_revenue.rename("revenue_KES").to_frame())
print(f"Best: {best_month.strftime('%B')} — KES {monthly_revenue.max():,.0f}")
print(f"Worst: {worst_month.strftime('%B')} — KES {monthly_revenue.min():,.0f}")
print(f"Best exceeds worst by {month_difference_pct:.2f}%")

,revenue_KES
order_date,
2026-01,11786600
2026-02,11772000
2026-03,12396800
2026-04,13518300
2026-05,11415400
2026-06,10903400


Best: April — KES 13,518,300
Worst: June — KES 10,903,400
Best exceeds worst by 23.98%


**Answer:** April is best at KES 13,518,300; June is worst at KES 10,903,400. April exceeds June by KES 2,614,900, or 23.98% relative to June. This is a descriptive comparison, not proof of a seasonal pattern; June’s generated coverage ends before month-end.

## Task 7: Premium Online orders in Nairobi and Mombasa

Combine three conditions with &, copy the selected rows, and export without the DataFrame index.

In [7]:
premium_orders = sales.loc[
    sales["price_band"].eq("Premium")
    & sales["channel"].eq("Online")
    & sales["branch"].isin(["Nairobi", "Mombasa"])
].copy()
Path("outputs").mkdir(exist_ok=True)
premium_orders.to_csv("outputs/premium_online_coast_capital.csv", index=False)
print(f"Orders: {len(premium_orders):,}")
print(f"Revenue: KES {premium_orders.revenue.sum():,.0f}")
display(premium_orders.head())

Orders: 83
Revenue: KES 10,218,000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium


**Answer:** The filter returns 83 orders and KES 10,218,000 revenue, accounting for 14.23% of all revenue. Only Laptops and Tablets satisfy this unit-price definition of Premium; all exported orders are Online and belong to Nairobi or Mombasa.

## Task 8: Excel reconciliation

The workbook contains 1,000 source orders, formula-based revenue, SUMIFS branch totals, XLOOKUP regions and a native branch-by-channel PivotTable. Read cached XLSX results with standard-library zipfile/XML (no additional Excel reader library) and compare every branch and channel cell with pandas.

In [8]:
NS = {"s": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
with zipfile.ZipFile("excel/duka_branch_summary.xlsx") as archive:
    string_list = []
    if "xl/sharedStrings.xml" in archive.namelist():
        strings = ET.fromstring(archive.read("xl/sharedStrings.xml"))
        string_list = ["".join(node.itertext()) for node in strings]
    sheet = ET.fromstring(archive.read("xl/worksheets/sheet1.xml"))
    cached = {}
    formulas = {}
    for cell in sheet.findall(".//s:sheetData/s:row/s:c", NS):
        address, kind = cell.attrib["r"], cell.get("t")
        value = cell.find("s:v", NS)
        if kind == "s":
            cached[address] = string_list[int(value.text)]
        elif kind == "inlineStr":
            cached[address] = "".join(cell.find("s:is", NS).itertext())
        elif value is not None and value.text is not None:
            cached[address] = value.text if kind in ("str", "e") else float(value.text)
        if cell.find("s:f", NS) is not None:
            formulas[address] = cell.find("s:f", NS).text
    excel_branch = pd.Series({cached[f"A{i}"]: cached[f"C{i}"] for i in range(6, 11)})
    pandas_branch = sales.groupby("branch").revenue.sum()
    assert excel_branch.sort_index().equals(pandas_branch.astype(float).sort_index())
    expected_pivot = sales.pivot_table(index="branch", columns="channel", values="revenue", aggfunc="sum", fill_value=0)
    for row in range(17, 22):
        for col in ["B", "C", "D"]:
            assert cached[f"{col}{row}"] == expected_pivot.loc[cached[f"A{row}"], cached[f"{col}16"]]
    assert cached["C11"] == cached["E22"] == sales.revenue.sum()
    assert "SUMIFS" in formulas["C6"] and "XLOOKUP" in formulas["B6"]
    native_pivots = [path for path in archive.namelist() if path.startswith("xl/pivotTables/") and path.endswith(".xml") and "/_rels/" not in path]
    assert native_pivots
    pivot_definition = ET.fromstring(archive.read(native_pivots[0]))
    assert pivot_definition.find("s:rowFields/s:field", NS).get("x") == "2"
    assert pivot_definition.find("s:colFields/s:field", NS).get("x") == "5"
    assert pivot_definition.find("s:dataFields/s:dataField", NS).get("fld") == "7"
reconciliation = pd.DataFrame({"pandas": pandas_branch, "Excel_SUMIFS": excel_branch})
reconciliation["difference"] = reconciliation.Excel_SUMIFS - reconciliation.pandas
display(reconciliation)
print(f"SUMIFS and native PivotTable totals match pandas: KES {sales.revenue.sum():,.0f}")
print("All 15 branch/channel cells match; XLOOKUP formulas and native PivotTable parts verified.")

,pandas,Excel_SUMIFS,difference
Eldoret,8844200,8844200.0,0.0
Kisumu,10868100,10868100.0,0.0
Mombasa,14003700,14003700.0,0.0
Nairobi,26106100,26106100.0,0.0
Nakuru,11970400,11970400.0,0.0


SUMIFS and native PivotTable totals match pandas: KES 71,792,500
All 15 branch/channel cells match; XLOOKUP formulas and native PivotTable parts verified.


**Answer:** Excel’s branch SUMIFS total and PivotTable grand total both equal **KES 71,792,500**, with zero difference for each branch and all 15 branch/channel intersections. The Branches mapping follows the Practicals: Nairobi–Central, Mombasa–Coast, Kisumu–Western, Nakuru–Rift Valley and Eldoret–North Rift. Source edits recalculate SUMIFS/XLOOKUP; refresh the PivotTable after changing Sales.

## Task 9: Management conclusion

The five-sentence Executive Summary is directly after the title and is copied unchanged into README.md. Revenue measures sales, not profit; this dataset has no cost or margin data. Prioritise Nairobi and the identified Premium Online segment, investigate April’s product/channel mix before repeating campaigns, and compare future complete periods before drawing seasonal conclusions.